---

# ~ DAC Find IT! 2026: Advanced Face Anti-Spoofing ~
*6-Class Face Liveness Detection*

---

## Table of Contents

1. [**Prapemrosesan Data & Augmentasi Tingkat Lanjut**](#1)
2. [**Membangun Arsitektur Swin dan CNN**](#2)
3. [**Strategi Pelatihan (Warm-up & Loss)**](#3)
4. [**Kalibrasi Probabilitas (Thresholding)**](#4)
5. [**Prediksi TTA & Pengumpulan (Submission)**](#5)

---

## Libraries

In [1]:
!pip install -q timm albumentations opencv-python-headless tqdm scikit-learn

In [2]:
# 1. Pustaka Sistem & Manipulasi Data Dasar
import os                     # Interaksi dengan sistem file (membuat folder, membaca path)
import gc                     # Garbage Collector: Membersihkan RAM/VRAM yang bocor
import warnings               # Mengelola peringatan sistem
import random                 # Generator angka acak dasar

import numpy as np            # Komputasi numerik dan manipulasi matriks tingkat tinggi
import pandas as pd           # Manipulasi data tabular (CSV, DataFrames)

# 2. Pustaka Computer Vision Tradisional
import cv2                    # OpenCV: Membaca, menulis, dan memanipulasi gambar (BGR ke RGB)

# 3. Pustaka PyTorch (Deep Learning Engine)
import torch                  # Tensor library (inti dari PyTorch)
import torch.nn as nn         # Neural Network (Layer, Loss Functions)
import torch.optim as optim   # Algoritma Optimasi (AdamW, SGD, Scheduler)
import torch.nn.functional as F # Fungsi aktivasi tanpa parameter (Softmax, ReLU)
from torch.utils.data import Dataset, DataLoader # Manajemen antrean batch data

# 4. Pustaka Arsitektur Model Canggih (State-of-the-Art)
import timm                   # PyTorch Image Models: Gudang arsitektur (Swin, ConvNeXt, EfficientNet)

# 5. Pustaka Augmentasi Gambar (Kelas Berat)
import albumentations as A    # Library augmentasi tercepat untuk CV (Noise, Blur, Dropout)
from albumentations.pytorch import ToTensorV2 # Mengubah format gambar HWC ke format Tensor PyTorch CHW

# 6. Pustaka Machine Learning Klasik & Evaluasi
from sklearn.model_selection import StratifiedKFold # Pembagian data CV yang seimbang (Class Balanced)
from sklearn.metrics import f1_score, classification_report # Evaluasi metrik performa model
from scipy.optimize import minimize # Algoritma pencarian bobot optimal (Metode Powell)

# 7. Pustaka Utilitas Interaktif
from tqdm.auto import tqdm    # Menampilkan progress bar interaktif saat training/inferensi

# --- Pengaturan Lingkungan (Environment Constraints) ---
warnings.filterwarnings("ignore") # Sembunyikan pesan deprecation agar output bersih
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True' # Cegah fragmentasi memori pada GPU

---

# Prapemrosesan Data & Augmentasi <a name="1"></a>

---

Tantangan terbesar dalam mendeteksi *spoofing* wajah (seperti topeng silikon, layar ponsel, atau cetakan kertas) adalah kecenderungan model untuk menghafal *noise* latar belakang atau spesifikasi kamera, alih-alih fitur keaslian wajah itu sendiri. 

Untuk mencapai akurasi maksimal dan generalisasi yang tangguh, notebook ini menerapkan dua strategi utama di tahap awal:
1. **Stratified K-Fold Cross Validation:** Memastikan distribusi 6 kelas selalu seimbang di setiap lipatan eksperimen.
2. **Advanced Data Augmentation:** Mensimulasikan artefak kompresi, variasi pencahayaan ekstrem, dan distorsi sensor (*ISONoise/GaussNoise*) agar model dipaksa mempelajari tekstur mikro dari objek, bukan resolusi gambar aslinya.

## Import & Global Configuration 

In [3]:
class CFG:
    """Konfigurasi sentral untuk seluruh hyperparameter"""
    seed = 42
    img_size = 384
    batch_size = 8
    n_folds = 5
    num_classes = 6       # FIX: Tambahkan jumlah kelas
    num_workers = 2       # FIX: Tambahkan jumlah worker untuk DataLoader
    base_path = "/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026"
    train_dir = os.path.join(base_path, "train")
    # Lebih rapi jika device juga dideklarasikan di sini sejak awal
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Dataframe creation & stratified K-FOLD

In [4]:
# Mapping direktori ke Label
class_names = sorted(os.listdir(CFG.train_dir))
label2id = {class_name: i for i, class_name in enumerate(class_names)}
id2label = {i: class_name for class_name, i in label2id.items()}

data = []
for class_name in class_names:
    class_dir = os.path.join(CFG.train_dir, class_name)
    if os.path.isdir(class_dir):
        for img_name in os.listdir(class_dir):
            data.append({
                "image_path": os.path.join(class_dir, img_name),
                "label": label2id[class_name]
            })

df_all = pd.DataFrame(data)

# Menerapkan Stratified K-Fold untuk distribusi kelas yang adil
skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
df_all['fold'] = -1

for fold, (train_idx, valid_idx) in enumerate(skf.split(X=df_all, y=df_all['label'])):
    df_all.loc[valid_idx, 'fold'] = fold

print(f"Total Data: {len(df_all)} gambar.")
display(df_all.groupby(['fold', 'label']).size().unstack(fill_value=0))

Total Data: 1652 gambar.


label,0,1,2,3,4,5
fold,,,,,,
0,45,56,23,46,68,93
1,45,56,23,46,68,93
2,45,55,24,46,67,93
3,45,55,24,46,67,93
4,44,56,24,45,68,93


## Pipeline Augmentasi 

In [5]:
# Transformasi Latih: Dioptimalkan untuk tantangan Anti-Spoofing
train_transforms = A.Compose([
    A.RandomResizedCrop(size=(CFG.img_size, CFG.img_size), scale=(0.8, 1.0), p=1.0),
    A.HorizontalFlip(p=0.5),
    
    # Blok 1: Simulasi artefak kamera digital & sensor noise
    A.OneOf([
        A.ImageCompression(p=1.0), 
        A.GaussianBlur(blur_limit=(3, 7), p=1.0),                       
        A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=1.0), 
        A.GaussNoise(p=1.0),                    
    ], p=0.5),
    
    # Blok 2: Pertahanan terhadap over/under exposure pada layar ponsel/kertas
    A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.3),
    
    # Blok 3: Distorsi spasial dan Oklusi (Meniru kondisi in-the-wild)
    A.Affine(scale=(0.95, 1.05), translate_percent=(-0.05, 0.05), rotate=(-15, 15), p=0.5),
    A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(8, 32), hole_width_range=(8, 32), fill=0, p=0.5),
    
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# Transformasi Validasi: Representasi murni gambar asli
valid_transforms = A.Compose([
    A.Resize(height=CFG.img_size, width=CFG.img_size),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

## Kelas pytorch dataset

In [6]:
class SpoofingDataset(Dataset):
    def __init__(self, df, transforms=None):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'image_path']
        image = cv2.imread(img_path)
        
        # Proteksi jika gambar korup atau jalur tidak ditemukan
        if image is None:
            image = np.zeros((CFG.img_size, CFG.img_size, 3), dtype=np.uint8)
        else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transforms:
            augmented = self.transforms(image=image)
            image = augmented['image']

        label = self.df.loc[idx, 'label']
        return image, torch.tensor(label, dtype=torch.long)

---

# Membangun Arsitektur Swin dan CNN <a name="2"></a>

---

Dalam tantangan *Face Anti-Spoofing* (FAS), sebuah model tidak bisa hanya mengandalkan satu cara pandang. Topeng silikon 3D mungkin memiliki tekstur yang sangat mirip dengan kulit asli (menipu CNN), sedangkan foto di layar HP mungkin memiliki proporsi wajah yang sempurna (menipu Transformer). 

Untuk mengatasi hal ini, kita membangun **Arsitektur Heterogen** yang terdiri dari dua model dengan cara kerja yang saling melengkapi:

#### 1. Swin Transformer (`swin_base_patch4`)
Swin Transformer bekerja dengan membagi gambar menjadi *patches* dan mencari hubungan antar area (*Shifted Window Attention*). Model ini bertindak sebagai "Mata Makro".
* **Fungsi Utama:** Mendeteksi anomali geometris, kesalahan pencahayaan global, dan proporsi yang tidak wajar (biasa ditemukan pada manekin atau topeng).
* **Modifikasi Khusus:** Kita membuang *head* klasifikasi bawaan dan menggantinya dengan **Custom Head (Linear → BatchNorm → ReLU → Dropout)**. `ReLU` memberikan batasan non-linear yang lebih tajam dibandingkan fungsi aktivasi bawaan, sangat cocok untuk membedakan fitur *spoofing* yang halus.
* **Optimasi:** Mengaktifkan `Gradient Checkpointing` agar kita tetap bisa menggunakan gambar beresolusi tinggi (384x384) tanpa mengalami *Out of Memory* (OOM) pada VRAM.

#### 2. EfficientNet V2 (`tf_efficientnetv2_m`)
EfficientNet adalah arsitektur berbasis Konvolusi (CNN) murni yang menggunakan *filter* bergeser. Model ini bertindak sebagai "Kaca Pembesar".
* **Fungsi Utama:** Sangat sensitif terhadap anomali tekstur tingkat piksel (*Local Receptive Field*). Sangat andal dalam mendeteksi **Pola Moiré** (garis-garis aneh saat memotret layar HP) atau serat kertas pada *printed spoof*.

## Model Architectures 

In [7]:
# Pastikan CFG.device sudah terdefinisi di sel sebelumnya
if not hasattr(CFG, 'device'):
    CFG.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SwinWithCustomHead(nn.Module):
    """Swin Transformer dengan Custom Head ReLU untuk klasifikasi 6-kelas."""
    def __init__(self, model_name, num_classes=6):
        super(SwinWithCustomHead, self).__init__()
        
        # Load backbone tanpa classification head bawaan
        self.backbone = timm.create_model(
            model_name, 
            pretrained=True, 
            num_classes=0,       
            drop_rate=0.3,       
            attn_drop_rate=0.2   
        )
        
        # Optimasi VRAM: Gradient Checkpointing
        self.backbone.set_grad_checkpointing(enable=True)
        
        num_features = self.backbone.num_features
        
        # Custom head menggunakan ReLU sesuai spesifikasi riset
        self.custom_head = nn.Sequential(
            nn.Linear(num_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        output = self.custom_head(features)
        return output

def create_swin_model():
    """Membuat instance Swin Transformer Base 384ms."""
    model_name = 'swin_base_patch4_window12_384.ms_in22k'
    return SwinWithCustomHead(model_name, num_classes=6).to(CFG.device)

def create_cnn_model():
    """Membuat instance EfficientNetV2 Medium."""
    model_name = 'tf_efficientnetv2_m.in21k_ft_in1k'
    model = timm.create_model(
        model_name, 
        pretrained=True, 
        num_classes=6,
        drop_rate=0.3,       
        drop_path_rate=0.2   
    )
    return model.to(CFG.device)

## Model Varification

In [8]:
def verify_architectures():
  
    
    # Dummy input (Batch=2, Channel=3, Size=384x384)
    dummy_input = torch.randn(2, 3, 384, 384).to(CFG.device)
    
    # Inisialisasi model secara temporer
    swin_temp = create_swin_model()
    cnn_temp = create_cnn_model()
    
    with torch.no_grad():
        out_swin = swin_temp(dummy_input)
        out_cnn = cnn_temp(dummy_input)
    
    print(f" Swin Output Shape : {out_swin.shape}")
    print(f" CNN Output Shape  : {out_cnn.shape}")
    
    # Bersihkan memori GPU secara total
    del swin_temp, cnn_temp, dummy_input
    torch.cuda.empty_cache()
    gc.collect()

verify_architectures()

model.safetensors:   0%|          | 0.00/451M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/218M [00:00<?, ?B/s]

 Swin Output Shape : torch.Size([2, 6])
 CNN Output Shape  : torch.Size([2, 6])


---

# Strategi Pelatihan (Warm-up & Loss) <a name="3"></a>

---

Melatih model *Face Anti-Spoofing* bukanlah sekadar menurunkan nilai *loss* secepat mungkin. Model sangat rentan terhadap **Overconfidence** (terlalu percaya diri pada tebakannya) dan **Class Imbalance** pada fitur yang sulit dikenali (*hard-examples*). Oleh karena itu, kita menerapkan dua strategi utama:

1. **Focal Loss + Label Smoothing:** Alih-alih menggunakan *Cross-Entropy* biasa, kita menggunakan *Focal Loss* untuk memaksa model fokus belajar dari gambar yang sering salah ditebak (misal: membedakan kulit asli vs cetakan kertas resolusi tinggi). *Label Smoothing* (0.1) mencegah model memberikan probabilitas mutlak (100%), yang terbukti ampuh membuat batas keputusan (*decision boundary*) menjadi lebih toleran dan general.

2. **Cosine Annealing Learning Rate:**
   Kita menggunakan *scheduler* yang menurunkan *Learning Rate* (LR) secara perlahan menyerupai kurva kosinus. Ini membantu model masuk ke titik *local minima* yang lebih luas dan stabil, meningkatkan skor validasi di tahap akhir pelatihan.

3. **Automatic Mixed Precision (AMP):**
   Pelatihan dipercepat hingga 2x lipat dan penggunaan VRAM dipangkas menggunakan `torch.amp`, memungkinkan kita melatih model berat di GPU Kaggle tanpa *Out of Memory* (OOM).

## Custom Loss Function

In [9]:
class FocalLossWithSmoothing(nn.Module):
    """
    Kombinasi Focal Loss (fokus pada sampel sulit) 
    dan Label Smoothing (mencegah overconfidence).
    """
    def __init__(self, alpha=1.0, gamma=2.0, smoothing=0.1, reduction='mean'):
        super(FocalLossWithSmoothing, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smoothing = smoothing
        self.reduction = reduction

    def forward(self, inputs, targets):
        num_classes = inputs.size(1)
        
        # 1. Terapkan Label Smoothing
        smooth_target = F.one_hot(targets, num_classes=num_classes).float()
        smooth_target = smooth_target * (1.0 - self.smoothing) + (self.smoothing / num_classes)
        
        # 2. Hitung Focal Loss
        log_pt = F.log_softmax(inputs, dim=1)
        pt = torch.exp(log_pt)
        
        ce_loss = -(smooth_target * log_pt).sum(dim=1)
        focal_term = self.alpha * (1 - pt.gather(1, targets.view(-1, 1)).squeeze(1)) ** self.gamma
        
        loss = focal_term * ce_loss
        
        if self.reduction == 'mean':
            return loss.mean()
        return loss.sum()

## Training Loop Pelatihan 10-Model Ensembel

In [10]:
def train_fold(fold, train_loader, valid_loader, model_func, model_prefix, epochs=20, lr=1e-5):
    print(f"\n{'='*40}\n MELATIH: {model_prefix.upper()} | FOLD {fold}\n{'='*40}")
    
    model = model_func().to(CFG.device)
    criterion = FocalLossWithSmoothing(gamma=2.0, smoothing=0.1) 
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-7)
    scaler = torch.amp.GradScaler('cuda') # AMP untuk hemat VRAM
    
    best_val_f1 = 0.0
    patience, patience_counter = 5, 0
    save_path = f"best_{model_prefix}_fold_{fold}.pth"

    for epoch in range(epochs):
        # 1. Fase Pelatihan (Train)
        model.train()
        train_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Ep {epoch+1}/{epochs} [TRAIN]", leave=False)
        
        for images, labels in pbar:
            images, labels = images.to(CFG.device), labels.to(CFG.device)
            optimizer.zero_grad()
            
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # Proteksi gradien meledak
            scaler.step(optimizer)
            scaler.update()
            
            train_loss += loss.item() * images.size(0)
            
        avg_train_loss = train_loss / len(train_loader.dataset)
        
        # 2. Fase Validasi (Eval)
        model.eval()
        val_loss, all_preds, all_labels = 0.0, [], []
        with torch.no_grad():
            for images, labels in valid_loader:
                images, labels = images.to(CFG.device), labels.to(CFG.device)
                with torch.amp.autocast('cuda'):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                    
                val_loss += loss.item() * images.size(0)
                all_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                
        avg_val_loss = val_loss / len(valid_loader.dataset)
        val_macro_f1 = f1_score(all_labels, all_preds, average='macro')
        scheduler.step()
        
        print(f"Ep {epoch+1:02d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val F1: {val_macro_f1:.4f}")
        
        # 3. Simpan Model Terbaik & Early Stopping
        if val_macro_f1 > best_val_f1:
            best_val_f1 = val_macro_f1
            torch.save(model.state_dict(), save_path)
            patience_counter = 0
        else:
            patience_counter += 1
            
        if patience_counter >= patience:
            print(f"Early Stopping dipicu! Model mulai overfitting.")
            break
            
    # Pembersihan VRAM agar siap untuk model berikutnya
    del model, optimizer, scaler, criterion
    torch.cuda.empty_cache(); gc.collect()
    return best_val_f1

# --- BAGIAN B: EKSEKUSI LOOP 10-MODEL ---
scores = {"swin": [], "cnn": []}

print(f"\n Memulai Proses Pelatihan 10-Model Ensemble (5 Fold CV)...")

for fold in range(CFG.n_folds):
    print(f"\n\n{'='*50}\n MEMPERSIAPKAN DATA FOLD {fold}\n{'='*50}")
    
    # Memotong data spesifik untuk fold ini
    train_ds = SpoofingDataset(df_all[df_all['fold'] != fold], train_transforms)
    valid_ds = SpoofingDataset(df_all[df_all['fold'] == fold], valid_transforms)
    
    train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True, drop_last=True, num_workers=2)
    valid_loader = DataLoader(valid_ds, batch_size=CFG.batch_size, shuffle=False, num_workers=2)
    
    # Melatih Swin Transformer dilanjutkan EfficientNetV2
    for prefix, m_func in [("swin", create_swin_model), ("cnn", create_cnn_model)]:
        best_f1 = train_fold(
            fold=fold,
            train_loader=train_loader,
            valid_loader=valid_loader,
            model_func=m_func,
            model_prefix=prefix,
            epochs=20, 
            lr=1e-5 if prefix == "swin" else 2e-5 # CNN lebih stabil dengan LR sedikit lebih besar
        )
        scores[prefix].append(best_f1)

# --- BAGIAN C: REKAPITULASI AKHIR ---
print(f"\n{'='*50}\n REKAPITULASI 10-MODEL ENSEMBLE\n{'='*50}")
for model_type, model_scores in scores.items():
    print(f"--- {model_type.upper()} ---")
    for i, s in enumerate(model_scores): 
        print(f"Fold {i} F1: {s:.4f}")
    print(f"Rata-rata {model_type.upper()}: {np.mean(model_scores):.4f}\n")


 Memulai Proses Pelatihan 10-Model Ensemble (5 Fold CV)...


 MEMPERSIAPKAN DATA FOLD 0

 MELATIH: SWIN | FOLD 0


Ep 1/20 [TRAIN]:   0%|          | 0/165 [00:00<?, ?it/s]

Ep 01 | Train Loss: 1.1396 | Val Loss: 0.6866 | Val F1: 0.6805


Ep 2/20 [TRAIN]:   0%|          | 0/165 [00:00<?, ?it/s]

Ep 02 | Train Loss: 0.7277 | Val Loss: 0.4743 | Val F1: 0.7531


...

Ep 14 | Train Loss: 0.4944 | Val Loss: 0.5591 | Val F1: 0.7811


Ep 15/20 [TRAIN]:   0%|          | 0/165 [00:00<?, ?it/s]

Ep 15 | Train Loss: 0.4290 | Val Loss: 0.5535 | Val F1: 0.7700


Ep 16/20 [TRAIN]:   0%|          | 0/165 [00:00<?, ?it/s]

Ep 16 | Train Loss: 0.4272 | Val Loss: 0.5480 | Val F1: 0.7854
Early Stopping dipicu! Model mulai overfitting.

 REKAPITULASI 10-MODEL ENSEMBLE
--- SWIN ---
Fold 0 F1: 0.8139
Fold 1 F1: 0.8406
Fold 2 F1: 0.8245
Fold 3 F1: 0.8605
Fold 4 F1: 0.8616
Rata-rata SWIN: 0.8402

--- CNN ---
Fold 0 F1: 0.7772
Fold 1 F1: 0.7428
Fold 2 F1: 0.7410
Fold 3 F1: 0.7843
Fold 4 F1: 0.7864
Rata-rata CNN: 0.7663



## Report 

In [11]:
print(f"\n{'='*50}\nREKAPITULASI 10-MODEL ENSEMBLE\n{'='*50}")
for model_type, model_scores in scores.items():
    print(f"--- {model_type.upper()} ---")
    for i, s in enumerate(model_scores): print(f"Fold {i} F1: {s:.4f}")
    print(f"Rata-rata: {np.mean(model_scores):.4f}\n")


REKAPITULASI 10-MODEL ENSEMBLE
--- SWIN ---
Fold 0 F1: 0.8139
Fold 1 F1: 0.8406
Fold 2 F1: 0.8245
Fold 3 F1: 0.8605
Fold 4 F1: 0.8616
Rata-rata: 0.8402

--- CNN ---
Fold 0 F1: 0.7772
Fold 1 F1: 0.7428
Fold 2 F1: 0.7410
Fold 3 F1: 0.7843
Fold 4 F1: 0.7864
Rata-rata: 0.7663



---

# Kalibrasi Probabilitas (Thresholding) <a name="4"></a>

---

Model *Deep Learning* sering kali mengalami masalah **Kalibrasi Probabilitas**. Misalnya, model mungkin sangat "pelit" memberikan probabilitas tinggi pada kelas minoritas seperti `fake_unknown`, sehingga kelas tersebut selalu kalah saat kita melakukan fungsi `argmax`. 

Tahap ini dirancang untuk memperbaiki kelemahan tersebut:
1. **Ekstraksi OOF (Out-of-Fold):** Kita akan mengumpulkan prediksi dari 5-Fold model menggunakan data validasi (data yang tidak pernah dilihat model saat *training*).
2. **Class-Weight Calibration (Powell Method):** Kita menggunakan algoritma optimasi SciPy untuk mencari 6 angka pengali (satu untuk setiap kelas). Algoritma ini akan mencari kombinasi pengali yang memberikan **Macro F1-Score tertinggi** pada data OOF.

## OOF Inference Engine 

In [12]:
def extract_oof_probs(model_func, path_template, data_df):
    """Fungsi umum untuk mengambil prediksi probabilitas dari model."""
    oof_probs = np.zeros((len(data_df), CFG.num_classes))
    
    for fold in range(CFG.n_folds):
        val_idx = data_df[data_df['fold'] == fold].index
        valid_loader = DataLoader(
            SpoofingDataset(data_df.iloc[val_idx], transforms=valid_transforms), 
            batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers
        )
        
        model = model_func().to(CFG.device)
        model.load_state_dict(torch.load(path_template.format(fold), map_location=CFG.device))
        model.eval()
        
        fold_preds = []
        with torch.no_grad():
            for images, _ in valid_loader:
                images = images.to(CFG.device)
                probs = F.softmax(model(images), dim=1)
                fold_preds.append(probs.cpu().numpy())
        
        oof_probs[val_idx] = np.vstack(fold_preds)
        
        # Cleanup
        del model; torch.cuda.empty_cache(); gc.collect()
        
    return oof_probs

## Execution: oof extraction 

In [13]:
# Ekstraksi probabilitas untuk kedua arsitektur
oof_probs_swin = extract_oof_probs(create_swin_model, "best_swin_fold_{}.pth", df_all)
oof_probs_cnn  = extract_oof_probs(create_cnn_model, "best_cnn_fold_{}.pth", df_all)
oof_labels     = df_all['label'].values

# Ensemble Awal: 60% Swin, 40% CNN
oof_probs_ensemble = (oof_probs_swin * 0.60) + (oof_probs_cnn * 0.40)

## Optimization & Calibration 

In [14]:
# Evaluasi Awal
preds_before = np.argmax(oof_probs_ensemble, axis=1)
print("ENSEMBLE PERFORMANCE (PRE-CALIBRATION)")
print(classification_report(oof_labels, preds_before, target_names=class_names, digits=4))

# Fungsi Optimasi
def optimize_f1(weights):
    calibrated = oof_probs_ensemble * weights
    return -1.0 * f1_score(oof_labels, np.argmax(calibrated, axis=1), average='macro')

# Menjalankan Minimizer
res = minimize(
    optimize_f1, x0=[1.0]*6, method='Powell', 
    bounds=[(0.5, 2.0)]*6
)

best_weights = res.x
final_probs = oof_probs_ensemble * best_weights
preds_final = np.argmax(final_probs, axis=1)

print("OPTIMIZED PERFORMANCE (POST-CALIBRATION)")
print(classification_report(oof_labels, preds_final, target_names=class_names, digits=4))
print(f"\n Best Class-Weights to use in Submission:\n{list(np.round(best_weights, 4))}")

ENSEMBLE PERFORMANCE (PRE-CALIBRATION)
                precision    recall  f1-score   support

fake_mannequin     0.8565    0.8795    0.8678       224
     fake_mask     0.8530    0.8561    0.8546       278
  fake_printed     0.6850    0.7373    0.7102       118
   fake_screen     0.9188    0.7904    0.8498       229
  fake_unknown     0.8234    0.8550    0.8389       338
    realperson     0.8739    0.8796    0.8767       465

      accuracy                         0.8481      1652
     macro avg     0.8351    0.8330    0.8330      1652
  weighted avg     0.8504    0.8481    0.8484      1652

OPTIMIZED PERFORMANCE (POST-CALIBRATION)
                precision    recall  f1-score   support

fake_mannequin     0.8477    0.9196    0.8822       224
     fake_mask     0.8897    0.8417    0.8651       278
  fake_printed     0.6500    0.7712    0.7054       118
   fake_screen     0.9385    0.7991    0.8632       229
  fake_unknown     0.8212    0.8698    0.8448       338
    realperson     0

---

# Prediksi TTA & Pengumpulan (Submission) <a name="4"></a>

---

Tahap terakhir adalah melakukan prediksi pada dataset kompetisi. Untuk mendapatkan skor yang kompetitif, kita tidak hanya melakukan prediksi satu kali, melainkan menggunakan dua teknik kunci:

1. **Test-Time Augmentation (TTA):** Kita melakukan prediksi dua kali untuk setiap gambar—satu pada gambar asli dan satu pada gambar yang dibalik secara horizontal (*flipped*). Rata-rata dari keduanya membantu model mengabaikan bias orientasi wajah.
2. **Weighted 10-Model Ensemble:** Kita menggabungkan probabilitas dari 5 model Swin dan 5 model CNN dengan rasio bobot yang telah ditentukan (misal: 85% Swin, 15% CNN) untuk menyeimbangkan pemahaman konteks global dan tekstur lokal.
3. **Post-Calibration:** Menerapkan bobot kelas dari hasil optimasi Powell (Tahap 4) untuk memaksimalkan Macro F1-Score pada papan peringkat (*Leaderboard*).

## Inference Dataset & TTA preaparation 

In [15]:
# Load Submission Template
test_dir = os.path.join(CFG.base_path, "test")
sub_df = pd.read_csv(os.path.join(CFG.base_path, "samplesubmission.csv"))

class TestDatasetTTA(Dataset):
    def __init__(self, df, test_dir):
        self.df = df
        self.test_dir = test_dir
        self.transforms = A.Compose([
            A.Resize(height=CFG.img_size, width=CFG.img_size),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])
        # Mapping file untuk mencegah error jika ekstensi berbeda
        self.existing_files = {os.path.splitext(f)[0]: f for f in os.listdir(test_dir) if os.path.isfile(os.path.join(test_dir, f))}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = str(self.df.iloc[idx]['id'])
        img_id_clean = os.path.splitext(img_id)[0] 
        
        # Load Image (dengan proteksi anti-crash)
        if img_id_clean in self.existing_files:
            img_path = os.path.join(self.test_dir, self.existing_files[img_id_clean])
            image = cv2.imread(img_path) 
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) if image is not None else np.zeros((CFG.img_size, CFG.img_size, 3), dtype=np.uint8)
        else:
            image = np.zeros((CFG.img_size, CFG.img_size, 3), dtype=np.uint8)
        
        # TTA: Original & Flipped
        img_orig = self.transforms(image=image)['image']
        img_flip = self.transforms(image=cv2.flip(image, 1))['image']

        return img_id, img_orig, img_flip

test_loader = DataLoader(
    TestDatasetTTA(sub_df, test_dir), 
    batch_size=CFG.batch_size, 
    shuffle=False, 
    num_workers=CFG.n_workers if hasattr(CFG, 'n_workers') else 2
)

## OOF inference Engine & Ensemble 

In [16]:
def run_tta_inference(model_func, model_prefix, weight_ratio):
    """Mengeksekusi inferensi TTA 5-Fold untuk arsitektur tertentu."""
    print(f"\n Eksekusi {model_prefix.upper()} TTA 5-Fold (Bobot: {weight_ratio*100:.0f}%)")
    
    ensemble_probs = np.zeros((len(sub_df), 6))
    
    for fold in range(CFG.n_folds):
        model = model_func().to(CFG.device)
        model.load_state_dict(torch.load(f'best_{model_prefix}_fold_{fold}.pth', map_location=CFG.device))
        model.eval()

        fold_probs = []
        with torch.no_grad():
            for _, imgs_orig, imgs_flip in tqdm(test_loader, desc=f"Fold {fold}"):
                # TTA Averaging
                p_orig = F.softmax(model(imgs_orig.to(CFG.device)), dim=1)
                p_flip = F.softmax(model(imgs_flip.to(CFG.device)), dim=1)
                fold_probs.append(((p_orig + p_flip) / 2.0).cpu().numpy())

        # Rata-rata dari K-Fold
        ensemble_probs += np.vstack(fold_probs) / CFG.n_folds
        
        del model; torch.cuda.empty_cache(); gc.collect()
        
    return ensemble_probs * weight_ratio

# Eksekusi Inferensi Modular
swin_weighted_probs = run_tta_inference(create_swin_model, "swin", weight_ratio=0.85)
cnn_weighted_probs  = run_tta_inference(create_cnn_model, "cnn", weight_ratio=0.15)

# Penggabungan 10-Model
final_ensemble_probs = swin_weighted_probs + cnn_weighted_probs

# Terapkan Bobot Powell (Ganti dengan array hasil keluaran Stage 4 Anda jika ada)
best_powell_weights = np.array([1.0, 1.0, 1.0, 1.0, 1.0, 1.0]) 
calibrated_probs = final_ensemble_probs * best_powell_weights


 Eksekusi SWIN TTA 5-Fold (Bobot: 85%)


Fold 0:   0%|          | 0/51 [00:00<?, ?it/s]

Fold 1:   0%|          | 0/51 [00:00<?, ?it/s]

Fold 2:   0%|          | 0/51 [00:00<?, ?it/s]

Fold 3:   0%|          | 0/51 [00:00<?, ?it/s]

...

Fold 0:   0%|          | 0/51 [00:00<?, ?it/s]

Fold 1:   0%|          | 0/51 [00:00<?, ?it/s]

Fold 2:   0%|          | 0/51 [00:00<?, ?it/s]

Fold 3:   0%|          | 0/51 [00:00<?, ?it/s]

Fold 4:   0%|          | 0/51 [00:00<?, ?it/s]

## Submission Generation 

In [17]:
final_preds = np.argmax(calibrated_probs, axis=1)

# Mapping Label
class_names = ['fake_mannequin', 'fake_mask', 'fake_printed', 'fake_screen', 'fake_unknown', 'realperson']
id2label = {i: name for i, name in enumerate(class_names)}

sub_df['label'] = [id2label[p] for p in final_preds]
sub_df.to_csv('submission.csv', index=False)